# Exercise 3: Search Evaluation — Precision@K and Recall@K


The goal: Be able to evaluate topic search results. Are search results on topic? 

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..')
sys.path.insert(0, str(REPO_ROOT / 'src'))

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from search import InMemorySearcher

OUTPUT_PATH = REPO_ROOT / 'output'

In [ ]:
# Load shared resources
print('Loading topic model...')
topic_model = BERTopic.load(str(OUTPUT_PATH / 'bertopic_model'))

print('Loading posts...')
with open(OUTPUT_PATH / 'doc_index.json') as f:
    doc_index = json.load(f)
posts = list(doc_index.values())

print('Loading topic assignments...')
assignments = pd.read_csv(OUTPUT_PATH / 'topic_assignments.csv')

print('Initializing searcher...')
searcher = InMemorySearcher(posts)

topic_info = topic_model.get_topic_info()
topic_ids = [t for t in topic_info['Topic'].tolist() if t != -1]
print(f'\nReady. Topics: {topic_ids}')

---
## Helper: evaluate search results

This function is pre-filled. It searches for each topic, then measures
how many of the top-k results are assigned to that topic by BERTopic.

In [ ]:
def evaluate(topic_embeddings: dict, top_k: int = 20, label: str = "") -> pd.DataFrame:
    """Search each topic and compute match ratio against topic_assignments.csv."""
    rows = []
    for topic_id, embedding in topic_embeddings.items():
        results = searcher.search_similar_documents(np.array(embedding), top_k=top_k)
        result_ids = {r['post_id'] for r in results}

        assigned = set(assignments.loc[assignments['topic_id'] == topic_id, 'post_id'])
        matched = len(result_ids & assigned)
        match_ratio = matched / top_k

        keywords = ', '.join(w for w, _ in topic_model.get_topic(topic_id)[:5])
        rows.append({
            'topic_id': topic_id,
            'keywords': keywords,
            f'{label}matched': matched,
            f'{label}match_ratio': round(match_ratio, 2),
        })

    df = pd.DataFrame(rows).set_index('topic_id')
    mean = df[f'{label}match_ratio'].mean()
    print(f"Mean match ratio ({label.strip('_') or 'result'}): {mean:.2f}")
    return df

---
## Part A — Naive topic centroid embedding

BERTopic stores a centroid for each topic in `topic_model.topic_embeddings_`.
Index 0 is the outlier cluster (-1); topic 0 is at index 1, topic 1 at index 2, etc.

> **Before running:** What match ratio do you expect?
> Write your prediction here: `~___`

In [ ]:
# TODO: build a dict {topic_id: centroid_embedding} for all topic_ids
# Hint: topic_model.topic_embeddings_[topic_id + 1]  (index 0 = outlier topic -1)

naive_embeddings = {}  # replace with your code

print(f"Built embeddings for {len(naive_embeddings)} topics")

In [ ]:
assert len(naive_embeddings) == len(topic_ids), "Build an embedding for every topic_id"
print("✓ naive_embeddings built")

In [ ]:
results_naive = evaluate(naive_embeddings, top_k=20, label='naive_')
results_naive

---
## Part B — Localized keyword embedding

**Step 1:** Open `exercises/02_topic_model.py` and complete the TODOs in `save_localized_embeddings()`.

**Step 2:** Run the pipeline from the terminal:
```bash
python exercises/02_topic_model.py
```
This saves `output/topic_embeddings_localized.json`.

**Step 3:** Come back and run the cells below.

> **Before running:** Do you expect the match ratio to go up or down? By how much?

In [ ]:
localized_path = OUTPUT_PATH / 'topic_embeddings_localized.json'

if not localized_path.exists():
    print("⚠  output/topic_embeddings_localized.json not found.")
    print("   Complete Step 1 & 2 above, then re-run this cell.")
else:
    with open(localized_path) as f:
        raw = json.load(f)
    localized_embeddings = {int(k): np.array(v) for k, v in raw.items()}
    print(f"Loaded localized embeddings for {len(localized_embeddings)} topics")

In [ ]:
results_localized = evaluate(localized_embeddings, top_k=20, label='localized_')
results_localized

---
## Side-by-side comparison

In [ ]:
comparison = results_naive[['keywords', 'naive_match_ratio']].join(
    results_localized[['localized_match_ratio']]
)
comparison['delta'] = comparison['localized_match_ratio'] - comparison['naive_match_ratio']
comparison.loc['MEAN'] = [
    '',
    comparison['naive_match_ratio'].mean().round(2),
    comparison['localized_match_ratio'].mean().round(2),
    comparison['delta'].mean().round(2),
]
comparison

---
## Discussion

- Why does the localized embedding perform better?
- What would happen if the keywords were noisy or domain-specific?
- How would you extend this to user-generated topics?

**Launch the Streamlit app to explore interactively:**
```bash
streamlit run app.py
```